# Agents

In [1]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [2]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [3]:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
	index=index,
	llm_client=openai_client,
	instructions=instructions,
)

In [4]:
search_tool = {
	"type": "function",
	"function": {
		"name": "search",
		"description": "Search the FAQ database for entries matching the given query.",
		"parameters": {
			"type": "object",
			"properties": {
				"query": {
					"type": "string",
					"description": "Search query text to look up in the course FAQ."
				}
			},
			"required": ["query"],
			"additionalProperties": False
		}
	}
}

In [5]:
def search(query):
	boost_dict = {"question": 3.0, "section": 0.5}
	filter_dict = {"course": "llm-zoomcamp"}

	return index.search(
		query,
		num_results=5,
		boost_dict=boost_dict,
		filter_dict=filter_dict
	)

In [6]:
messages = [
  {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

In [7]:
response = openai_client.chat.completions.create(
	model="gpt-4o-mini",
	messages=messages,
	user="llm-zoomcamp",
	stream=False,
	tools=[search_tool]
)

In [8]:
import json

if response.choices[0].finish_reason == "tool_calls":
	message = response.choices[0].message
	function_call = response.choices[0].message.tool_calls[0].function
  
	if function_call.name == "search":
		# retrieve tool call params and call search function
		args = json.loads(function_call.arguments)
		results = search(**args)
		result_json = json.dumps(results, indent=2)
		
		# add the model response and tool call results to the message history
		messages.append(message)
		messages.append({
			"role": "tool",
			"tool_call_id": response.choices[0].message.tool_calls[0].id,
			"content": result_json
		})

		# send new prompt with updated message history
		response = openai_client.chat.completions.create(
			model="gpt-4o-mini",
			messages=messages,
			user="llm-zoomcamp",
			stream=False,
			tools=[search_tool]
		)

		print(response.choices[0].message.content)
	else:
		print("Unable to perform search")
else:
	print(response.choices[0].message.content)

Yes, you can still join the course! However, if you want to receive a certificate, you'll need to submit your project while submissions are still being accepted. Feel free to start learning and submitting homework right away!


## Agentic loop

In [9]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [10]:
def make_call(call):
	if call.function.name == "search":
		args = json.loads(call.function.arguments)
		results = search(**args)
		result_json = json.dumps(results, indent=2)

		return {
			"role": "tool",
			"tool_call_id": call.id,
			"content": result_json
		}

In [11]:
question = "I just discovered the course. Can I join it?"

In [ ]:
def agent_loop(instructions, question, model="gpt-4o-mini") -> str:
	messages = [
		{"role": "developer", "content": instructions},
		{"role": "user", "content": question},
	]

	last_answer = "Could not generate a response"
	it = 1

	while True:
		print(f"iteration #{it}...")	
		has_function_calls = False
		
		response = openai_client.chat.completions.create(
			model=model,
			messages=messages,
			user="llm-zoomcamp",
			stream=False,
			tools=[search_tool]
		)

		message = response.choices[0].message
		messages.append(message)
		
		for item in response.choices:
			if item.finish_reason == "tool_calls":
				tool_call = message.tool_calls[0]
				print("function_call:", tool_call.function.name, tool_call.function.arguments)
				call_output = make_call(tool_call)
				messages.append(call_output)
				has_function_calls = True
			elif item.finish_reason == "stop":
				last_answer = item.message.content
				print(item.message.content)
		
		it += 1
		if has_function_calls == False:
			break

	return last_answer

In [13]:
agent_loop(instructions, "How do I run Olama locally?")

f'iteration #1...
function_call: search {"query":"run Olama locally"}
f'iteration #2...
function_call: search {"query":"Olama local setup instructions"}
f'iteration #3...
function_call: search {"query":"Olama local installation requirements"}
f'iteration #4...
ChatCompletionMessage(content="To run Olama locally, you'll generally need to follow these steps based on information collected from the course materials regarding local setups.\n\n1. **Environment Setup**: You need to ensure that you have a suitable environment set up on your local machine. This typically includes:\n   - Python installed (version requirements can vary)\n   - Necessary libraries (you may need libraries like `uv`, `docker`, etc.)\n   - A text editor or IDE for writing your code (e.g., VS Code, PyCharm).\n\n2. **Local Installation of Olama**: \n   - **Docker**: Ensure Docker is installed, as many modern applications depend on containerized environments.\n   - **Clone the Repository**: Get the codebase from which yo

"To run Olama locally, you'll generally need to follow these steps based on information collected from the course materials regarding local setups.\n\n1. **Environment Setup**: You need to ensure that you have a suitable environment set up on your local machine. This typically includes:\n   - Python installed (version requirements can vary)\n   - Necessary libraries (you may need libraries like `uv`, `docker`, etc.)\n   - A text editor or IDE for writing your code (e.g., VS Code, PyCharm).\n\n2. **Local Installation of Olama**: \n   - **Docker**: Ensure Docker is installed, as many modern applications depend on containerized environments.\n   - **Clone the Repository**: Get the codebase from which you want to run Olama; it might be hosted on platforms like GitHub.\n   - **Dependencies**: Check if there’s a `requirements.txt` or `docker-compose.yml` file in your cloned repository to install all necessary dependencies. You can typically run:\n     ```bash\n     pip install -r requirement